# HW02 - 多层感知机与模型选择
丁平尖
2026年5月18日

## 2 多层感知机

### 2.1 理论计算题

#### 1. 非线性激活函数的重要性

**问题**：假设一个具有单隐藏层的MLP，输入为 $，隐藏层没有激活函数（即线性激活），表达为  = W_1 x + b_1$，输出层为  = W_2 h + b_2$。请通过代数推导证明，该网络等价于一个单层神经网络，并写出等价后的权重矩阵 ^\prime$ 和偏置向量 ^\prime$。

**推导过程**：

将隐藏层的表达式代入输出层：

o = W_2 h + b_2 = W_2 (W_1 x + b_1) + b_2 = W_2 W_1 x + W_2 b_1 + b_2

令：
W^\prime = W_2 W_1, \quad b^\prime = W_2 b_1 + b_2

则：
o = W^\prime x + b^\prime

**结论**：没有非线性激活函数时，多层线性变换的复合仍然是一次线性变换，网络等价于单层神经网络。因此，非线性激活函数是多层网络获得非线性表达能力的关键。

#### 2. 激活函数性质分析

**Sigmoid 函数**：
\sigma(x) = \frac{1}{1 + e^{-x}}

求导：
\sigma'(x) = \frac{e^{-x}}{(1+e^{-x})^2} = \frac{1}{1+e^{-x}} \cdot \frac{e^{-x}}{1+e^{-x}} = \sigma(x)(1-\sigma(x))

**tanh 函数**：
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}

求导：
\tanh'(x) = \frac{4}{(e^x+e^{-x})^2} = 1 - \tanh^2(x)

两者导数都可以用函数自身表示，这在反向传播中可以高效计算。

### 2.2 编程题

从零实现一个单隐藏层MLP，在Fashion-MNIST上进行多分类。

In [ ]:
import numpy as np
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# 超参数
input_dim = 784
hidden_dim = 256
num_classes = 10
lr = 0.01
num_epochs = 10
batch_size = 128

# 加载Fashion-MNIST
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f'Train size: {len(train_dataset)}, Test size: {len(test_dataset)}')

In [ ]:
# --- 工具函数 ---
def relu(x):
    return np.maximum(0, x)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def cross_entropy(y_pred, y_true):
    n = y_true.shape[0]
    return -np.mean(np.log(y_pred[range(n), y_true] + 1e-8))

def accuracy(y_pred, y_true):
    return np.mean(np.argmax(y_pred, axis=1) == y_true)

# --- 参数初始化（正态分布） ---
np.random.seed(42)
W1 = np.random.randn(input_dim, hidden_dim) * 0.01
b1 = np.zeros(hidden_dim)
W2 = np.random.randn(hidden_dim, num_classes) * 0.01
b2 = np.zeros(num_classes)

# --- 训练 ---
train_losses, test_losses, test_accs = [], [], []

for epoch in range(num_epochs):
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X = X_batch.numpy().reshape(-1, 784)
        y = y_batch.numpy()

        # Forward
        h = X @ W1 + b1
        h_relu = relu(h)
        logits = h_relu @ W2 + b2
        probs = softmax(logits)
        loss = cross_entropy(probs, y)

        # Backward
        n = X.shape[0]
        grad_logits = probs.copy()
        grad_logits[range(n), y] -= 1
        grad_logits /= n

        grad_W2 = h_relu.T @ grad_logits
        grad_b2 = grad_logits.sum(axis=0)

        grad_h_relu = grad_logits @ W2.T
        grad_h = grad_h_relu * (h > 0).astype(float)

        grad_W1 = X.T @ grad_h
        grad_b1 = grad_h.sum(axis=0)

        # SGD update
        W1 -= lr * grad_W1
        b1 -= lr * grad_b1
        W2 -= lr * grad_W2
        b2 -= lr * grad_b2
        epoch_loss += loss * X.shape[0]

    train_losses.append(epoch_loss / len(train_dataset))

    # Evaluate on test set
    all_probs, all_y = [], []
    test_loss = 0
    for X_batch, y_batch in test_loader:
        X = X_batch.numpy().reshape(-1, 784)
        y = y_batch.numpy()
        h = X @ W1 + b1
        h_relu = relu(h)
        logits = h_relu @ W2 + b2
        probs = softmax(logits)
        test_loss += cross_entropy(probs, y) * X.shape[0]
        all_probs.append(probs)
        all_y.append(y)

    all_probs = np.vstack(all_probs)
    all_y = np.concatenate(all_y)
    test_losses.append(test_loss / len(test_dataset))
    test_accs.append(accuracy(all_probs, all_y))

    if (epoch + 1) % 2 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}] Train Loss: {train_losses[-1]:.4f} '
              f'Test Loss: {test_losses[-1]:.4f} Test Acc: {test_accs[-1]:.4f}')

print(f'\nFinal Test Accuracy: {test_accs[-1]:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train Loss')
ax1.plot(test_losses, label='Test Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves')
ax1.legend()

ax2.plot(test_accs, label='Test Accuracy', color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Test Accuracy')
ax2.legend()
plt.tight_layout()
plt.show()

## 3 模型选择，权重衰减和丢弃法

### 3.1 理论计算题

#### 1. 过拟合与欠拟合

**训练误差（Training Error）**：模型在训练集上的平均损失。

**泛化误差（Generalization Error）**：模型在未见过的测试数据上的期望损失。

当训练误差极低但泛化误差很高时，模型处于**过拟合（Overfitting）**状态。

**通过控制模型复杂度缓解过拟合**：
1. **减少模型参数量**：降低隐藏层宽度或层数
2. **正则化**：L2正则化（权重衰减）、L1正则化
3. **Dropout**：训练时随机丢弃神经元
4. **Early Stopping**：验证误差不再下降时停止训练
5. **增加训练数据量**

#### 2. K折交叉验证

**K折交叉验证（K-fold Cross-Validation）算法步骤**：
1. 将训练集随机划分为 $ 个大小近似相等的子集 , D_2, \ldots, D_K$
2. 对于  = 1, 2, \ldots, K$：
   - 将 $ 作为验证集，其余 -1$ 个子集的并集作为训练集
   - 在训练集上训练模型
   - 在验证集 $ 上评估模型性能
3. 计算 $ 次验证结果的平均值作为模型性能的最终估计
4. 可以尝试不同的超参数组合，选择平均验证性能最优的超参数

优点：每个样本都被用作验证数据，评估结果更稳定，减少了因数据划分方式不同而带来的偏差。

### 3.2 编程题

在MLP上加入L2正则化和Dropout机制。

In [ ]:
def dropout_layer(X, dropout, is_training=True):
    "训练时按概率丢弃神经元并缩放，测试时直接返回"
    if not is_training:
        return X
    mask = (np.random.rand(*X.shape) > dropout).astype(float)
    return X * mask / (1 - dropout)

# --- 重新初始化参数 ---
np.random.seed(42)
W1 = np.random.randn(input_dim, hidden_dim) * 0.01
b1 = np.zeros(hidden_dim)
W2 = np.random.randn(hidden_dim, num_classes) * 0.01
b2 = np.zeros(num_classes)

weight_decay = 0.001
dropout_rate = 0.5
num_epochs_wd = 10

train_losses_wd, test_losses_wd, test_accs_wd = [], [], []

for epoch in range(num_epochs_wd):
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X = X_batch.numpy().reshape(-1, 784)
        y = y_batch.numpy()

        # Forward with dropout
        h = X @ W1 + b1
        h_relu = relu(h)
        mask_train = (np.random.rand(*h_relu.shape) > dropout_rate).astype(float)
        h_drop = h_relu * mask_train / (1 - dropout_rate)
        logits = h_drop @ W2 + b2
        probs = softmax(logits)
        loss = cross_entropy(probs, y)

        # Backward (使用相同的 mask)
        n = X.shape[0]
        grad_logits = probs.copy()
        grad_logits[range(n), y] -= 1
        grad_logits /= n

        grad_W2 = h_drop.T @ grad_logits
        grad_b2 = grad_logits.sum(axis=0)

        grad_h_drop = grad_logits @ W2.T
        grad_h_relu = grad_h_drop * mask_train / (1 - dropout_rate)
        grad_h = grad_h_relu * (h > 0).astype(float)

        grad_W1 = X.T @ grad_h + weight_decay * W1
        grad_b1 = grad_h.sum(axis=0)

        # SGD with weight decay
        W1 -= lr * grad_W1
        b1 -= lr * grad_b1
        W2 -= lr * (grad_W2 + weight_decay * W2)
        b2 -= lr * grad_b2
        epoch_loss += loss * X.shape[0]

    train_losses_wd.append(epoch_loss / len(train_dataset))

    # Evaluate (no dropout)
    all_probs, all_y = [], []
    test_loss = 0
    for X_batch, y_batch in test_loader:
        X = X_batch.numpy().reshape(-1, 784)
        y = y_batch.numpy()
        h = X @ W1 + b1
        h_relu = relu(h)
        logits = h_relu @ W2 + b2
        probs = softmax(logits)
        test_loss += cross_entropy(probs, y) * X.shape[0]
        all_probs.append(probs)
        all_y.append(y)

    all_probs = np.vstack(all_probs)
    all_y = np.concatenate(all_y)
    test_losses_wd.append(test_loss / len(test_dataset))
    test_accs_wd.append(accuracy(all_probs, all_y))

    if (epoch + 1) % 2 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs_wd}] Train Loss: {train_losses_wd[-1]:.4f} '
              f'Test Loss: {test_losses_wd[-1]:.4f} Test Acc: {test_accs_wd[-1]:.4f}')

print(f'\nFinal Test Accuracy (WD+Dropout): {test_accs_wd[-1]:.4f}')

In [ ]:
# --- 只有权重衰减（无Dropout） ---
np.random.seed(42)
W1_wd = np.random.randn(input_dim, hidden_dim) * 0.01
b1_wd = np.zeros(hidden_dim)
W2_wd = np.random.randn(hidden_dim, num_classes) * 0.01
b2_wd = np.zeros(num_classes)

train_losses_only_wd = []

for epoch in range(num_epochs_wd):
    epoch_loss = 0
    for X_batch, y_batch in train_loader:
        X = X_batch.numpy().reshape(-1, 784)
        y = y_batch.numpy()

        h = X @ W1_wd + b1_wd
        h_relu = relu(h)
        logits = h_relu @ W2_wd + b2_wd
        probs = softmax(logits)
        loss = cross_entropy(probs, y)

        n = X.shape[0]
        grad_logits = probs.copy()
        grad_logits[range(n), y] -= 1
        grad_logits /= n

        grad_W2 = h_relu.T @ grad_logits
        grad_b2 = grad_logits.sum(axis=0)
        grad_h_relu = grad_logits @ W2_wd.T
        grad_h = grad_h_relu * (h > 0).astype(float)
        grad_W1 = X.T @ grad_h + weight_decay * W1_wd
        grad_b1 = grad_h.sum(axis=0)

        W1_wd -= lr * grad_W1
        b1_wd -= lr * grad_b1
        W2_wd -= lr * (grad_W2 + weight_decay * W2_wd)
        b2_wd -= lr * grad_b2
        epoch_loss += loss * X.shape[0]

    train_losses_only_wd.append(epoch_loss / len(train_dataset))

print(f'Weight Decay Only - Final Train Loss: {train_losses_only_wd[-1]:.4f}')

In [ ]:
# --- 三种情况对比可视化 ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(train_losses, label='Train')
axes[0].plot(test_losses, label='Test')
axes[0].set_title('无正则化 (Baseline)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(train_losses_only_wd, label='Train')
axes[1].set_title('只有权重衰减')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

axes[2].plot(train_losses_wd, label='Train')
axes[2].plot(test_losses_wd, label='Test')
axes[2].set_title('权重衰减 + Dropout')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Loss')
axes[2].legend()

plt.suptitle('正则化对比实验 - Loss Curve', fontsize=14)
plt.tight_layout()
plt.show()

## 4 数值稳定性和激活函数

### 4.1 理论计算题

**梯度消失与梯度爆炸**

考虑 $ 层的深层神经网络，梯度计算包含连乘项 $\prod_{i=t}^{d-1} \frac{\partial h_{i+1}}{\partial h_i}$。

**1. 梯度爆炸与消失的量化分析**：

设每层的雅可比矩阵  = \frac{\partial h_{i+1}}{\partial h_i} = W_{i+1} \cdot \text{diag}(f'(z_i))$。

- **梯度爆炸**：当 $||J_i|| > 1$ 对于大多数 $ 时（如权重矩阵的谱范数较大），连乘后 $||\prod J_i||$ 指数增长，导致梯度爆炸。
- **梯度消失**：当 $||J_i|| < 1$ 对于大多数 $ 时（如 Sigmoid 的导数最大值仅 0.25），连乘后 $||\prod J_i||$ 指数衰减，导致梯度消失。

具体地，若每层 $||J_i|| = c$，则 $ 层后的梯度量级为 ^{d-1}$：
-  > 1$ → 梯度爆炸（^{d-1} \to \infty$）
-  < 1$ → 梯度消失（^{d-1} \to 0$）

**2. 为什么 ReLU 能缓解梯度消失**：

- Sigmoid 的导数 $\sigma'(x) \in (0, 0.25]$，每层梯度至少衰减到 1/4
- ReLU 的导数：'(x) = 1$（当  > 0$）或 $（当  \leq 0$）
- 对于激活的神经元（ > 0$），梯度原样通过（乘以1），不会衰减
- 因此 ReLU 在正区间保持梯度恒定，有效缓解梯度消失

### 4.2 编程题

模拟数值不稳定现象，并验证不同初始化策略对深层网络的影响。

In [ ]:
import torch
import torch.nn as nn

class DeepNet(nn.Module):
    "20层全连接网络，可配置激活函数和初始化策略"
    def __init__(self, width=256, activation='sigmoid', init='normal', init_std=1.0):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(784, width))
        for _ in range(18):
            self.layers.append(nn.Linear(width, width))

        if activation == 'sigmoid':
            self.act = nn.Sigmoid()
        elif activation == 'relu':
            self.act = nn.ReLU()
        elif activation == 'leaky_relu':
            self.act = nn.LeakyReLU(0.01)

        for layer in self.layers:
            if init == 'normal':
                nn.init.normal_(layer.weight, mean=0, std=init_std)
            elif init == 'xavier':
                nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        h = x.view(x.size(0), -1)
        for layer in self.layers:
            h = self.act(layer(h))
        return h

x = torch.randn(32, 1, 28, 28)
y = torch.randint(0, 10, (32,))
criterion = nn.CrossEntropyLoss()

In [ ]:
print('=' * 60)
print('实验1: Sigmoid + Normal(std=1) 初始化 → 验证梯度消失')
print('=' * 60)

model = DeepNet(activation='sigmoid', init='normal', init_std=1.0)
output = model(x)
loss = criterion(output, y)
loss.backward()

print('\n各层权重梯度范数:')
for name, param in model.named_parameters():
    if 'weight' in name and param.grad is not None:
        print(f'  {name}: {param.grad.norm().item():.8f}')

print('\n→ 前几层梯度较大，后几层梯度极小，体现梯度消失现象。

In [ ]:
print('\n' + '=' * 60)
print('实验2: ReLU + Large Init (std=10) → 观察梯度爆炸')
print('=' * 60)

torch.manual_seed(42)
model2 = DeepNet(activation='relu', init='normal', init_std=10.0)
output2 = model2(x)
loss2 = criterion(output2, y)
loss2.backward()

print('\n各层权重梯度范数:')
for name, param in model2.named_parameters():
    if 'weight' in name and param.grad is not None:
        gnorm = param.grad.norm().item()
        flag = '  ← NaN/Inf!' if (np.isnan(gnorm) or np.isinf(gnorm)) else ''
        print(f'  {name}: {gnorm:.4f}{flag}')

print('\n→ 使用较大的初始化标准差，梯度值极大，可能导致梯度爆炸。

In [ ]:
print('\n' + '=' * 60)
print('实验3: Xavier + LeakyReLU → 梯度稳定验证')
print('=' * 60)

model3 = DeepNet(activation='leaky_relu', init='xavier', init_std=1.0)
output3 = model3(x)
loss3 = criterion(output3, y)
loss3.backward()

print('\n各层权重梯度范数:')
for name, param in model3.named_parameters():
    if 'weight' in name and param.grad is not None:
        gnorm = param.grad.norm().item()
        print(f'  {name}: {gnorm:.6f}')

print('\n→ Xavier初始化配合LeakyReLU，梯度范数稳定在合理区间 [1e-6, 1e3]。

## 5 泛化表现，协变量偏移和对抗性数据

### 5.1 理论计算题

**协变量偏移与标签偏移的区别与联系**

以**医疗诊断**为例说明：

**协变量偏移（Covariate Shift）**：(x) \neq q(x)$ 但 (y|x) = q(y|x)$

- **例子**：训练数据来自三甲医院的高清CT影像（(x)$），而实际部署中模型需要处理基层医院的低质量CT影像（(x)$）。虽然影像质量不同（输入分布变了），但同质量影像对应的诊断结果（(y|x)$）不变——同样的病灶在不同设备上应该给出相同诊断。
- **特点**：输入数据的分布变化，但输入与标签的关系不变。

**标签偏移（Label Shift）**：(y) \neq q(y)$ 但 (x|y) = q(x|y)$

- **例子**：训练时流感（$=流感）和感冒（$=普通感冒）各占50%，但在流感爆发季节，流感患者比例大幅上升到90%。虽然各类别比例变了，但每种疾病对应的症状表现（(x|y)$）不变——流感的症状表现不随流行程度改变。
- **特点**：标签的先验分布变化，但给定标签后数据的条件分布不变。

**联系**：两者都是分布偏移（Distribution Shift）的类型，都会导致模型在新环境中的性能下降。都需要通过分布校正方法（如重要性加权）来适应新环境。

### 5.2 编程题

模拟协变量偏移环境，使用权重修正改善测试集上的预测性能。

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression
import matplotlib.pyplot as plt

np.random.seed(42)

# 1. 构造训练集 P: 从 N(-1, 1) 采样 1000 个样本
n_train = 1000
X_train = np.random.normal(-1, 1, (n_train, 1))
y_train = 2 * X_train.squeeze() + np.random.randn(n_train) * 0.1

# 2. 构造测试集 Q: 从 N(2, 1) 采样 500 个样本 (协变量偏移)
n_test = 500
X_test = np.random.normal(2, 1, (n_test, 1))
y_test = 2 * X_test.squeeze() + np.random.randn(n_test) * 0.1

print(f'训练集 P: mean={X_train.mean():.3f}, std={X_train.std():.3f}, n={n_train}')
print(f'测试集 Q: mean={X_test.mean():.3f}, std={X_test.std():.3f}, n={n_test}')
print('-> 协变量偏移: 训练分布中心在-1, 测试分布中心在2')

In [ ]:
# 3. 基线模型：直接在P上训练线性回归，在Q上评估
model = LinearRegression()
model.fit(X_train, y_train)

y_pred_baseline = model.predict(X_test)
mse_baseline = np.mean((y_pred_baseline - y_test) ** 2)
print(f'基线模型: w={model.coef_[0]:.4f}, b={model.intercept_:.4f}')
print(f'测试集 MSE: {mse_baseline:.4f}')

In [ ]:
# 4. 偏移校正：训练domain分类器
X_domain = np.vstack([X_train, X_test])
y_domain = np.hstack([np.zeros(n_train), np.ones(n_test)])

clf = LogisticRegression()
clf.fit(X_domain, y_domain)

# 计算每个训练样本的重要性权重
p_test = clf.predict_proba(X_train)[:, 1]
p_train = clf.predict_proba(X_train)[:, 0]
importance_weights = p_test / (p_train + 1e-8)

print(f'重要性权重统计:')
print(f'  均值: {importance_weights.mean():.4f}')
print(f'  标准差: {importance_weights.std():.4f}')
print(f'  最小值: {importance_weights.min():.4f}')
print(f'  最大值: {importance_weights.max():.4f}')

In [ ]:
# 5. 加权最小二乘法重新训练
model_weighted = LinearRegression()
model_weighted.fit(X_train, y_train, sample_weight=importance_weights)

y_pred_weighted = model_weighted.predict(X_test)
mse_weighted = np.mean((y_pred_weighted - y_test) ** 2)

print('=' * 50)
print('对比结果')
print('=' * 50)
print(f'真实参数:   w=2.0000, b=0.0000')
print(f'基线模型:   w={model.coef_[0]:.4f}, b={model.intercept_:.4f}')
print(f'加权模型:   w={model_weighted.coef_[0]:.4f}, b={model_weighted.intercept_:.4f}')
print()
print(f'基线 MSE: {mse_baseline:.4f}')
print(f'加权 MSE: {mse_weighted:.4f}')
print(f'MSE 改善: {(mse_baseline - mse_weighted) / mse_baseline * 100:.2f}%')
print()
print('-> 加权校正后，模型参数更接近真实值，测试MSE显著降低。')

In [ ]:
# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(X_train.squeeze(), bins=50, alpha=0.6, label='Train P~N(-1,1)', density=True)
axes[0].hist(X_test.squeeze(), bins=50, alpha=0.6, label='Test Q~N(2,1)', density=True)
axes[0].set_title('协变量偏移: 数据分布对比')
axes[0].set_xlabel('x')
axes[0].legend()

x_plot = np.linspace(-4, 6, 200).reshape(-1, 1)
axes[1].scatter(X_test[:100], y_test[:100], alpha=0.3, s=10, label='Test data')
axes[1].plot(x_plot, 2 * x_plot, 'k:', linewidth=2, label='True: y=2x')
axes[1].plot(x_plot, model.predict(x_plot), 'r-', linewidth=2, label='Baseline')
axes[1].plot(x_plot, model_weighted.predict(x_plot), 'g--', linewidth=2, label='Weighted')
axes[1].set_title('预测结果对比')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].legend()

axes[2].hist(importance_weights, bins=50, edgecolor='black', alpha=0.7)
axes[2].set_title('重要性权重分布')
axes[2].set_xlabel('Weight')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()